# Strategy Exploration

This notebook explores the trading strategies used by the Paper Trading Bot.
We'll visualize indicators, backtest strategies, and compare their performance.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_bot.clients import get_data_client
from trading_bot.data.market_data import MarketDataFetcher
from trading_bot.data.indicators import (
    compute_sma, compute_ema, compute_rsi, compute_macd, compute_bollinger, compute_all_indicators
)
from trading_bot.backtest.engine import BacktestEngine
from trading_bot.strategies import get_strategy, STRATEGY_REGISTRY

## 1. Load Market Data

In [ ]:
client = get_data_client()
fetcher = MarketDataFetcher(client)

symbol = "AAPL"
bars = fetcher.get_bars(symbol, limit=365)
print(f"Loaded {len(bars)} bars for {symbol}")
bars.tail()

## 2. Visualize Technical Indicators

In [ ]:
df = compute_all_indicators(bars)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=[f"{symbol} Price + Bollinger Bands", "RSI", "MACD"])

# Price + SMAs + Bollinger
fig.add_trace(go.Candlestick(x=df.index, open=df['open'], high=df['high'],
                              low=df['low'], close=df['close'], name='Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['sma_20'], name='SMA 20', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['bb_upper'], name='BB Upper',
                          line=dict(width=1, dash='dash', color='gray')), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['bb_lower'], name='BB Lower',
                          line=dict(width=1, dash='dash', color='gray'),
                          fill='tonexty', fillcolor='rgba(128,128,128,0.1)'), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=df.index, y=df['rsi_14'], name='RSI'), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

# MACD
fig.add_trace(go.Scatter(x=df.index, y=df['macd'], name='MACD'), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['macd_signal'], name='Signal'), row=3, col=1)
colors = ['green' if v >= 0 else 'red' for v in df['macd_diff']]
fig.add_trace(go.Bar(x=df.index, y=df['macd_diff'], name='Histogram', marker_color=colors), row=3, col=1)

fig.update_layout(height=900, template='plotly_white', xaxis_rangeslider_visible=False)
fig.show()

## 3. Backtest All Strategies

In [ ]:
results = {}
for name in STRATEGY_REGISTRY:
    strategy = get_strategy(name)
    if name == 'ml':
        strategy.train(bars)
    engine = BacktestEngine(strategy)
    results[name] = engine.run(bars, symbol)
    r = results[name]
    print(f"{name:20s} | Return: {r.total_return_pct:+7.2f}% | Sharpe: {r.sharpe_ratio:6.2f} | "
          f"MaxDD: {r.max_drawdown:6.2f}% | WinRate: {r.win_rate:5.1f}% | Trades: {r.total_trades}")

## 4. Compare Equity Curves

In [ ]:
fig = go.Figure()
for name, result in results.items():
    if not result.equity_curve.empty:
        fig.add_trace(go.Scatter(
            x=result.equity_curve['timestamp'],
            y=result.equity_curve['equity'],
            mode='lines', name=name
        ))

fig.update_layout(
    title=f'Strategy Comparison — {symbol}',
    xaxis_title='Date', yaxis_title='Equity ($)',
    template='plotly_white', height=500
)
fig.show()

## 5. Performance Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Strategy': name,
        'Return %': f"{r.total_return_pct:+.2f}",
        'Sharpe Ratio': f"{r.sharpe_ratio:.2f}",
        'Max Drawdown %': f"{r.max_drawdown:.2f}",
        'Win Rate %': f"{r.win_rate:.1f}",
        'Total Trades': r.total_trades,
    }
    for name, r in results.items()
])
summary